In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/gemma-4-good-hackathon/NOTE.md


In [2]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 71.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 64.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 89.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 5.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently

In [3]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
import chromadb

print("All libraries imported successfully!")

All libraries imported successfully!


In [4]:
from kaggle_secrets import UserSecretsClient
import os

secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = secrets.get_secret("HF_token")  # lowercase t

model_id = "google/gemma-3-1b-it"

tokenizer = AutoTokenizer.from_pretrained(model_id, token=os.environ["HF_TOKEN"])
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    token=os.environ["HF_TOKEN"]
)

print("Gemma loaded successfully!")

config.json:   0%|          | 0.00/899 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.00G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

Gemma loaded successfully!


In [5]:
# Sample medical documents (we'll use real PubMed data later)
medical_docs = [
    "Type 2 diabetes is characterized by high blood sugar, insulin resistance, and lack of insulin. Symptoms include increased thirst, frequent urination, fatigue, and blurred vision.",
    "Early symptoms of kidney disease include swelling in legs and ankles, fatigue, shortness of breath, decreased urine output, and confusion.",
    "Hypertension or high blood pressure is a condition where blood pressure is consistently above 130/80 mmHg. It can lead to heart disease and stroke if untreated.",
    "Symptoms of heart attack include chest pain, shortness of breath, nausea, lightheadedness, and pain radiating to arm or jaw. Immediate medical attention is required.",
    "Asthma is a chronic lung condition causing airway inflammation. Symptoms include wheezing, coughing, chest tightness, and shortness of breath.",
    "Depression symptoms include persistent sadness, loss of interest, changes in appetite, sleep disturbances, fatigue, and difficulty concentrating.",
    "Anemia occurs when blood lacks enough healthy red blood cells. Symptoms include fatigue, weakness, pale skin, shortness of breath, and dizziness.",
    "Migraine is a neurological condition causing intense headaches, often with nausea, vomiting, and sensitivity to light and sound.",
]

# Load the embedding model (converts text to numbers for search)
print("Loading embedding model...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')
print("Embedding model loaded!")

# Create the vector database
chroma_client = chromadb.Client()
collection = chroma_client.create_collection("medical_docs")

# Convert documents to embeddings and store them
embeddings = embedder.encode(medical_docs).tolist()
collection.add(
    documents=medical_docs,
    embeddings=embeddings,
    ids=[f"doc_{i}" for i in range(len(medical_docs))]
)

print(f"Stored {len(medical_docs)} medical documents in the database!")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded!
Stored 8 medical documents in the database!


In [6]:
def medical_rag_agent(question):
    print(f"\nQuestion: {question}")
    print("-" * 50)
    
    # Step 1: Search documents
    print("Searching medical documents...")
    question_embedding = embedder.encode([question]).tolist()
    results = collection.query(
        query_embeddings=question_embedding,
        n_results=2
    )
    
    # Step 2: Get retrieved docs
    retrieved_docs = results['documents'][0]
    print(f"Found {len(retrieved_docs)} relevant documents")
    
    # Step 3: Build prompt
    context = "\n".join([f"- {doc}" for doc in retrieved_docs])
    prompt = f"""You are a helpful medical assistant. Answer the question using ONLY the context below. Give a single concise answer and end with one line saying 'Source: Medical Document Library'.

Context:
{context}

Question: {question}
Answer (be brief, end with source on one line):"""
    
    # Step 4: Generate answer
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            do_sample=False,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[-1]:],
        skip_special_tokens=True
    )
    
    # Clean up repeated lines
    lines = response.split('\n')
    seen = []
    for line in lines:
        if line not in seen:
            seen.append(line)
        if 'Source: Medical Document Library' in line:
            break
    
    clean_response = '\n'.join(seen)
    print("\nAnswer:")
    print(clean_response)
    return clean_response

print("RAG agent ready!")

RAG agent ready!


In [7]:
# Test your agent!
medical_rag_agent("What are the symptoms of diabetes?")

The following generation flags are not valid and may be ignored: ['top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.



Question: What are the symptoms of diabetes?
--------------------------------------------------
Searching medical documents...
Found 2 relevant documents

Answer:

The patient experiences increased thirst, frequent urination, fatigue, blurred vision, and swelling in legs and ankles. Source: Medical Document Library


'\nThe patient experiences increased thirst, frequent urination, fatigue, blurred vision, and swelling in legs and ankles. Source: Medical Document Library'

In [8]:
medical_rag_agent("What are the signs of a heart attack?")


Question: What are the signs of a heart attack?
--------------------------------------------------
Searching medical documents...
Found 2 relevant documents

Answer:

The patient experiences chest pain, shortness of breath, nausea, lightheadedness, and pain radiating to arm or jaw as symptoms of a heart attack. Source: Medical Document Library


'\nThe patient experiences chest pain, shortness of breath, nausea, lightheadedness, and pain radiating to arm or jaw as symptoms of a heart attack. Source: Medical Document Library'

In [9]:
medical_rag_agent("How is high blood pressure defined?")


Question: How is high blood pressure defined?
--------------------------------------------------
Searching medical documents...
Found 2 relevant documents

Answer:

High blood pressure is defined as consistently above 130/80 mmHg. Source: Medical Document Library


'\nHigh blood pressure is defined as consistently above 130/80 mmHg. Source: Medical Document Library'